# 04 — Outcome Model Analysis

Validate the WIN / LOSS / CANCELLED / EXPIRED model:
- Overall hit rate (target: 5-7%)
- Logistic S-curve shape (hit rate vs normalized spread)
- Affinity effect on hit rate
- Cancel-rho_k correlation (key Easter egg)
- Adverse selection: post-trade moves for informed vs uninformed


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit

from rfq_sim.utils.io import load_all
from rfq_sim.utils.diagnostics import (
    check_hit_rate, outcome_distribution, cancel_rho_correlation,
    affinity_hit_rate_monotonicity
)

data = load_all('../data')
obs  = data['observable']
gt   = data['ground_truth']
obs['timestamp'] = pd.to_datetime(obs['timestamp'])

print(f"Dataset: {len(obs):,} RFQs")
_ = check_hit_rate(obs)
_ = outcome_distribution(obs)


In [ ]:
# S-curve: hit rate as a function of normalized spread delta/delta0
# Should follow a logistic shape -- tighter quotes have higher hit rate

obs['norm_spread'] = obs['quote_delta'] / obs['spread_at_request'].clip(lower=0.01)
obs['win']         = (obs['outcome'] == 'WIN').astype(int)

# Bin by normalized spread
bins = pd.cut(obs['norm_spread'], bins=20)
s_curve = obs.groupby(bins, observed=True)['win'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
bin_centers = [float(str(b).split(',')[0].strip('(')) for b in s_curve.index]
ax.plot(bin_centers, s_curve.values, 'o-', color='steelblue', markersize=5)
ax.set_xlabel('Normalized Spread (delta / delta0)')
ax.set_ylabel('Hit Rate')
ax.set_title('Empirical S-Curve: Hit Rate vs Normalized Quote Spread\n'
             '(should decrease as spread increases)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/plots_s_curve.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Affinity effect on hit rate -- the core collaborative filtering signal
# Hit rate should increase with affinity quartile Q1 -> Q4

result = affinity_hit_rate_monotonicity(obs, gt)

fig, ax = plt.subplots(figsize=(7, 4))
result.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Affinity Quartile')
ax.set_ylabel('Hit Rate')
ax.set_title('Hit Rate by Affinity Quartile\n'
             '(monotonic increase = recommender has learnable signal)')
ax.axhline(obs['win'].mean(), color='red', linestyle='--', alpha=0.7, label='Overall mean')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cancel rate vs rho_k -- the adverse selection Easter egg
# Clients with high rho_k should cancel more, especially when
# prices move during the RFQ window

merged = obs.merge(gt[['rfq_id', 'rho_k']], on='rfq_id', how='left')
merged['cancelled'] = (merged['outcome'] == 'CANCELLED').astype(float)

rho_bins = pd.cut(merged['rho_k'], bins=5)
cancel_by_rho = merged.groupby(rho_bins, observed=True)['cancelled'].mean()

fig, ax = plt.subplots(figsize=(8, 4))
cancel_by_rho.plot(kind='bar', ax=ax, color='salmon', alpha=0.8)
ax.set_xlabel('Client Informativeness (rho_k)')
ax.set_ylabel('Cancel Rate')
ax.set_title('Cancel Rate by Client Informativeness (rho_k)\n'
             '(higher rho -> more cancellations = informed flow Easter egg)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/plots_cancel_rho.png', dpi=120, bbox_inches='tight')
plt.show()

corr = cancel_rho_correlation(obs, gt)


In [ ]:
# Post-trade adverse selection: informed moves vs noise
wins = obs[obs['outcome'] == 'WIN'].copy()
wins_gt = wins.merge(gt[['rfq_id', 'rho_k', 'informed_move']], on='rfq_id')

if len(wins_gt) > 10:
    informed   = wins_gt[wins_gt['informed_move'] == True]['post_trade_price_move']
    uninformed = wins_gt[wins_gt['informed_move'] == False]['post_trade_price_move']

    # Adverse direction: for buy (client buys, dealer sells short),
    # adverse move is price going UP. For sell it's going DOWN.
    # Here I just look at magnitude.
    if len(informed) > 0 and len(uninformed) > 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.hist(uninformed.dropna(), bins=30, alpha=0.6, label='Uninformed', color='steelblue', density=True)
        ax.hist(informed.dropna(),   bins=30, alpha=0.6, label='Informed',   color='salmon',   density=True)
        ax.axvline(0, color='black', linestyle='--', alpha=0.5)
        ax.set_xlabel('Post-Trade Price Move (30min)')
        ax.set_ylabel('Density')
        ax.set_title('Post-Trade Price Moves: Informed vs Uninformed Clients\n'
                     '(informed moves should be more adverse to dealer)')
        ax.legend()
        plt.tight_layout()
        plt.show()
